# 05 — Modelos a partir de especificação: a fábrica de **classes**

Até aqui a fábrica produzia **objetos**. Agora ela produz **classes**: um arquivo ([`configs/models.yaml`](configs/models.yaml))
descreve enums e modelos, um **meta-modelo** valida essa descrição, e quatro "alvos" materializam as classes:

```
configs/models.yaml ──► SpecFile (meta-modelo: valida a DEFINIÇÃO) ──┬──► Enum funcional (StrEnum / IntEnum)
                                                                     ├──► Pydantic create_model        (runtime)
                                                                     ├──► dataclasses.make_dataclass   (runtime, stdlib)
                                                                     └──► código-fonte gerado (.py)    (revisável, tipado)
```

Regra de negócio **não** mora no YAML: o arquivo só referencia normalizadores e validadores por nome, implementados em
[`validadores.py`](validadores.py).

In [1]:
import keyword
import pickle
from dataclasses import MISSING, asdict, field, make_dataclass
from datetime import date, datetime
from decimal import Decimal
from enum import IntEnum, StrEnum
from graphlib import CycleError, TopologicalSorter
from pathlib import Path
from typing import Annotated, Any, Literal, Self

import yaml
from pydantic import (UUID4, AfterValidator, BaseModel, BeforeValidator, ConfigDict, Field, Strict, TypeAdapter, ValidationError, create_model,
                      field_validator, model_validator)
from pydantic.alias_generators import to_camel

from validadores import NORMALIZERS, VALIDATORS

CONFIGS = Path("configs")
raw_spec = yaml.safe_load((CONFIGS / "models.yaml").read_text())
print(f"enums: {list(raw_spec['enums'])}")
print(f"models: {list(raw_spec['models'])}")

enums: ['BodyType', 'Priority']
models: ['Automobile', 'RegistrationCountry', 'ServiceTicket']


## 1. O meta-modelo: o que uma especificação pode dizer

Vocabulário **fechado**: tipos escalares de uma lista, nomes de enums/modelos do próprio arquivo, restrições compatíveis com o tipo,
validadores que existem no registro. Tudo o que estiver fora disso é erro **ao carregar o arquivo**.

In [2]:
SCALARS: dict[str, Any] = {"str": str, "int": int, "float": float, "bool": bool, "date": date, "datetime": datetime,
                           "decimal": Decimal, "uuid4": UUID4}
ORDERABLE = {"int", "float", "decimal", "date", "datetime"}

POLICIES: dict[str, dict[str, Any]] = {          # no projeto real, vem do arquivo de perfis do notebook 04
    "api_input": {"extra": "forbid", "str_strip_whitespace": True, "validate_assignment": True,
                  "alias_generator": "camel", "validate_by_name": True},
    "internal": {"extra": "forbid", "frozen": True, "strict": True},
}
ALIAS_GENERATORS = {"camel": to_camel}


def is_type_name(name: str) -> bool:
    return name.isidentifier() and name[0].isupper() and not keyword.iskeyword(name)


class EnumSpec(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    kind: Literal["str", "int"] = "str"
    values: list[str] | dict[str, int | str] = Field(min_length=1)

    @model_validator(mode="after")
    def coherent(self) -> Self:
        members = self.members()
        if len(set(members.values())) != len(members):
            raise ValueError("valores repetidos")
        bad = [name for name in members if not name.isidentifier()]
        if bad:
            raise ValueError(f"nomes de membro inválidos: {bad}")
        if self.kind == "int" and not all(isinstance(v, int) for v in members.values()):
            raise ValueError("kind=int exige valores inteiros (use o formato NOME: valor)")
        if self.kind == "str" and not all(isinstance(v, str) for v in members.values()):
            raise ValueError("kind=str exige valores texto")
        return self

    def members(self) -> dict[str, int | str]:
        if isinstance(self.values, list):
            return {value.upper(): value for value in self.values}
        return dict(self.values)


class FieldSpec(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    type: str
    many: bool = False
    nullable: bool = False
    default: Any = None
    description: str | None = None
    ge: Any = None
    le: Any = None
    min_length: int | None = Field(default=None, ge=0)
    max_length: int | None = Field(default=None, ge=1)
    pattern: str | None = None
    normalize: list[str] = []          # BeforeValidator, na ordem escrita
    validators: list[str] = []         # AfterValidator

    @property
    def required(self) -> bool:
        return "default" not in self.model_fields_set          # "default: null" é diferente de não ter default

    @model_validator(mode="after")
    def coherent(self) -> Self:
        if (self.ge is not None or self.le is not None) and self.type not in ORDERABLE:
            raise ValueError(f"ge/le não se aplicam a {self.type!r}")
        text_rules = [self.min_length, self.max_length, self.pattern]
        if any(rule is not None for rule in text_rules) and self.type != "str":
            raise ValueError(f"min_length/max_length/pattern só para str, não {self.type!r}")
        if self.pattern is not None and not (self.pattern.startswith("^") and self.pattern.endswith("$")):
            raise ValueError("pattern precisa ser ancorado (^...$): o Pydantic faz busca, não match completo")
        unknown = [n for n in self.normalize if n not in NORMALIZERS] + [n for n in self.validators if n not in VALIDATORS]
        if unknown:
            raise ValueError(f"funções desconhecidas {unknown}; normalize: {sorted(NORMALIZERS)}, validators: {sorted(VALIDATORS)}")
        if not self.required and self.default is None and not self.nullable:
            raise ValueError("default null exige nullable: true")
        return self


class ModelSpec(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    policy: Literal["api_input", "internal"] = "api_input"
    description: str | None = None
    fields: dict[str, FieldSpec] = Field(min_length=1)

    @field_validator("fields")
    @classmethod
    def python_names(cls, fields: dict[str, FieldSpec]) -> dict[str, FieldSpec]:
        bad = [name for name in fields if not name.isidentifier() or keyword.iskeyword(name) or name.startswith("_")]
        if bad:
            raise ValueError(f"nomes de campo inválidos em Python: {bad}")
        return fields


class SpecFile(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    enums: dict[str, EnumSpec] = {}
    models: dict[str, ModelSpec] = Field(min_length=1)

    @model_validator(mode="after")
    def resolvable(self) -> Self:
        names = list(self.enums) + list(self.models)
        bad = [name for name in names if not is_type_name(name)]
        if bad:
            raise ValueError(f"nomes de tipo devem ser identificadores em PascalCase: {bad}")
        clash = (set(self.enums) & set(self.models)) | (set(names) & set(SCALARS))
        if clash:
            raise ValueError(f"nomes repetidos entre enums/modelos/escalares: {sorted(clash)}")
        for model_name, model in self.models.items():
            for field_name, spec in model.fields.items():
                if spec.type not in SCALARS and spec.type not in self.enums and spec.type not in self.models:
                    raise ValueError(f"{model_name}.{field_name}: tipo desconhecido {spec.type!r}")
                if spec.type in self.models or spec.type in self.enums:
                    if spec.ge is not None or spec.le is not None:
                        raise ValueError(f"{model_name}.{field_name}: ge/le em tipo composto")
        self.model_order()                                      # levanta se houver ciclo
        return self

    def model_order(self) -> list[str]:
        graph = {name: {f.type for f in m.fields.values() if f.type in self.models} for name, m in self.models.items()}
        try:
            return list(TopologicalSorter(graph).static_order())
        except CycleError as ex:
            raise ValueError(f"referência circular entre modelos: {ex.args[1]}") from None


spec = SpecFile.model_validate(raw_spec)
print(f"ordem de construção: {spec.model_order()}")
print(f"Priority: {spec.enums['Priority'].members()}")
print(f"Automobile.number_of_doors.required: {spec.models['Automobile'].fields['number_of_doors'].required}")
print(f"Automobile.registration_country.required: {spec.models['Automobile'].fields['registration_country'].required}")

ordem de construção: ['RegistrationCountry', 'Automobile', 'ServiceTicket']
Priority: {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}
Automobile.number_of_doors.required: False
Automobile.registration_country.required: False


### O que o meta-modelo recusa

Cada item abaixo é um erro que, sem o meta-modelo, viraria um modelo errado em produção (ou uma exceção críptica no import):

In [3]:
def try_spec(label: str, patch: dict) -> None:
    candidate = {"enums": {"Color": {"values": ["red"]}}, "models": {"Item": {"fields": {"x": {"type": "str"}}}}}
    for key, value in patch.items():
        candidate[key] = value
    try:
        SpecFile.model_validate(candidate)
        print(f"{label}: aceito")
    except ValidationError as ex:
        error = ex.errors()[0]
        where = ".".join(map(str, error["loc"])) or "arquivo"
        print(f"{label}: {where} -> {error['msg'].removeprefix('Value error, ')}")


try_spec("tipo com erro de digitação", {"models": {"Item": {"fields": {"x": {"type": "strng"}}}}})
try_spec("chave com erro de digitação", {"models": {"Item": {"fields": {"x": {"type": "str", "mn_length": 2}}}}})
try_spec("pattern em int", {"models": {"Item": {"fields": {"x": {"type": "int", "pattern": "^1$"}}}}})
try_spec("pattern sem âncora", {"models": {"Item": {"fields": {"cep": {"type": "str", "pattern": "\\d{5}"}}}}})
try_spec("campo 'class'", {"models": {"Item": {"fields": {"class": {"type": "str"}}}}})
try_spec("default null sem nullable", {"models": {"Item": {"fields": {"x": {"type": "str", "default": None}}}}})
try_spec("normalizador inexistente", {"models": {"Item": {"fields": {"x": {"type": "str", "normalize": ["lower"]}}}}})
try_spec("enum com valor repetido", {"enums": {"Color": {"values": {"RED": "r", "ROSE": "r"}}}})
try_spec("modelo em snake_case", {"models": {"item": {"fields": {"x": {"type": "str"}}}}})
try_spec("ciclo A -> B -> A", {"models": {"A": {"fields": {"b": {"type": "B"}}}, "B": {"fields": {"a": {"type": "A"}}}}})

tipo com erro de digitação: arquivo -> Item.x: tipo desconhecido 'strng'
chave com erro de digitação: models.Item.fields.x.mn_length -> Extra inputs are not permitted
pattern em int: models.Item.fields.x -> min_length/max_length/pattern só para str, não 'int'
pattern sem âncora: models.Item.fields.cep -> pattern precisa ser ancorado (^...$): o Pydantic faz busca, não match completo
campo 'class': models.Item.fields -> nomes de campo inválidos em Python: ['class']
default null sem nullable: models.Item.fields.x -> default null exige nullable: true
normalizador inexistente: models.Item.fields.x -> funções desconhecidas ['lower']; normalize: ['strip_title', 'upper'], validators: ['no_digits']
enum com valor repetido: enums.Color -> valores repetidos
modelo em snake_case: arquivo -> nomes de tipo devem ser identificadores em PascalCase: ['item']
ciclo A -> B -> A: arquivo -> referência circular entre modelos: ['A', 'B', 'A']


> ⚠️ O meta-modelo garante **coerência** da definição, mas não tudo: um default que viola a restrição do próprio campo
> (`default: 9` com `le: 4`) só é detectável construindo o tipo. Os construtores abaixo checam isso.

## 2. Alvo 1 — Enums pela API funcional

In [4]:
def build_enums(spec: SpecFile) -> dict[str, type]:
    built = {}
    for name, enum_spec in spec.enums.items():
        base = IntEnum if enum_spec.kind == "int" else StrEnum
        built[name] = base(name, enum_spec.members())
    return built


enums = build_enums(spec)
BodyType, Priority = enums["BodyType"], enums["Priority"]
print(f"list(BodyType): {list(BodyType)}")
print(f"Priority(2): {Priority(2)!r}")
print(f"Priority.HIGH > Priority.LOW: {Priority.HIGH > Priority.LOW}")

list(BodyType): [<BodyType.SEDAN: 'sedan'>, <BodyType.COUPE: 'coupe'>, <BodyType.SUV: 'suv'>, <BodyType.TRUCK: 'truck'>]
Priority(2): <Priority.MEDIUM: 2>
Priority.HIGH > Priority.LOW: True


### Por que separar `normalize` de `validators`: ordem de execução

Duas regras do Pydantic decidem onde cada função entra. A primeira: restrições (`pattern`, `min_length`...) rodam **antes** de um
`AfterValidator`. Um normalizador como `upper` posto como *after* chega tarde demais:

In [5]:
VinAfter = Annotated[str, Field(pattern=r"^[A-Z0-9]{17}$"), AfterValidator(str.upper)]
VinBefore = Annotated[str, Field(pattern=r"^[A-Z0-9]{17}$"), BeforeValidator(lambda v: v.upper() if isinstance(v, str) else v)]
for label, vin_type in [("after", VinAfter), ("before", VinBefore)]:
    try:
        print(f"{label}: {TypeAdapter(vin_type).validate_python('1hgcm82633a004352')}")
    except ValidationError as ex:
        print(f"{label}: {ex.errors()[0]['type']}")

after: string_pattern_mismatch
before: 1HGCM82633A004352


A segunda: vários `BeforeValidator` rodam em **ordem inversa** à declarada. Por isso o construtor inverte a lista `normalize` —
quem escreve a especificação lê "primeiro A, depois B" e é isso que acontece. (Detalhe de acabamento: com o `Field(...)` declarado
**antes** do `BeforeValidator`, a mensagem de erro continua específica — "String should have at least 2 characters"; na ordem
inversa ela vira um genérico "Value should have at least 2 items after validation". O normalizador roda primeiro nos dois casos.)

In [6]:
trace = []
step = lambda name: BeforeValidator(lambda v: trace.append(name) or v)
TypeAdapter(Annotated[str, step("A"), step("B"), AfterValidator(lambda v: trace.append("after") or v)]).validate_python("x")
print(f"declarado A, B, after -> executado: {trace}")

declarado A, B, after -> executado: ['B', 'A', 'after']


## 3. Alvo 2 — Pydantic `create_model`

O mesmo "descritor de tipo" serve aos dois alvos de runtime: escalar/enum/modelo → restrições e validadores no elemento →
`list[...]` → `| None`.

In [7]:
def element_type(spec_field: FieldSpec, namespace: dict[str, Any]) -> Any:
    base = SCALARS.get(spec_field.type) or namespace[spec_field.type]
    constraints = {k: getattr(spec_field, k) for k in ("ge", "le", "min_length", "max_length", "pattern")
                   if getattr(spec_field, k) is not None}
    metadata = (
        ([Field(**constraints)] if constraints else [])           # Field antes: mensagens de erro específicas do tipo
        + [BeforeValidator(NORMALIZERS[n]) for n in reversed(spec_field.normalize)]   # before roda em ordem inversa
        + [AfterValidator(VALIDATORS[v]) for v in spec_field.validators]
    )
    return Annotated[(base, *metadata)] if metadata else base


def full_type(spec_field: FieldSpec, namespace: dict[str, Any]) -> Any:
    annotation = element_type(spec_field, namespace)
    if spec_field.many:
        annotation = list[annotation]
    if spec_field.nullable:
        annotation = annotation | None
    return annotation


def config_for(policy: str) -> ConfigDict:
    options = dict(POLICIES[policy])
    if "alias_generator" in options:
        options["alias_generator"] = ALIAS_GENERATORS[options["alias_generator"]]
    return ConfigDict(**options)


def checked_default(model: str, name: str, spec_field: FieldSpec, annotation: Any) -> Any:
    """Valida o default contra o próprio tipo e devolve o valor CONVERTIDO (ex.: 2 -> Priority.MEDIUM)."""
    if spec_field.default is None:
        return None
    try:
        return TypeAdapter(annotation).validate_python(spec_field.default)
    except ValidationError as ex:
        raise ValueError(f"{model}.{name}: default {spec_field.default!r} inválido: {ex.errors()[0]['msg']}") from None


def build_pydantic(spec: SpecFile) -> dict[str, type]:
    namespace: dict[str, Any] = build_enums(spec)
    for model_name in spec.model_order():
        model = spec.models[model_name]
        definitions = {}
        for name, spec_field in model.fields.items():
            annotation = full_type(spec_field, namespace)
            info = Field(description=spec_field.description) if spec_field.required else \
                Field(default=checked_default(model_name, name, spec_field, annotation), description=spec_field.description)
            definitions[name] = (annotation, info)
        namespace[model_name] = create_model(model_name, __config__=config_for(model.policy), __doc__=model.description,
                                             **definitions)
    return namespace


pyd = build_pydantic(spec)
ticket_data = {
    "vin": "1hgcm82633a004352",
    "openedAt": "2026-11-11T09:30:00",
    "car": {"manufacturer": "  bmw   motors ", "bodyType": "coupe", "manufacturedDate": "2023-01-01",
            "registrationCountry": {"name": "Brazil", "code3": "BRA"}, "topFeatures": ["teto solar"]},
}
ticket = pyd["ServiceTicket"].model_validate(ticket_data)
print(f"ticket.vin: {ticket.vin}")
print(f"ticket.priority: {ticket.priority!r}")
print(f"ticket.car.manufacturer: {ticket.car.manufacturer!r}")
print(f"ticket.car.registration_country: {ticket.car.registration_country!r}")

ticket.vin: 1HGCM82633A004352
ticket.priority: <Priority.MEDIUM: 2>
ticket.car.manufacturer: 'Bmw Motors'
ticket.car.registration_country: RegistrationCountry(name='Brazil', code3='BRA')


Repare em `ticket.priority`: o default da especificação é `2`, e o campo é `Priority`. Como o Pydantic **não valida defaults**
(`validate_default=False`), o construtor valida o default contra o próprio tipo e grava o valor **convertido** — senão o modelo
guardaria um `int` cru e o serializador avisaria (`PydanticSerializationUnexpectedValue`).

Validação completa, com as funções nomeadas (normalizadores `upper` no VIN e `strip_title` no fabricante; validador `no_digits` no país), aliases camelCase da política
`api_input` e o modelo `internal` (estrito, congelado) aninhado:

In [8]:
bad_ticket = {
    "vin": "123",
    "priority": 7,
    "openedAt": "ontem",
    "car": {"manufacturer": "B", "bodyType": "van", "manufacturedDate": "1979-12-31", "numberOfDoors": 5,
            "registrationCountry": {"name": "Bra2il", "code3": "br"}, "color": "azul"},
}
try:
    pyd["ServiceTicket"].model_validate(bad_ticket)
except ValidationError as ex:
    for error in ex.errors():
        print(f"{'.'.join(map(str, error['loc']))}: {error['msg']}")

vin: String should match pattern '^[A-HJ-NPR-Z0-9]{17}$'
priority: Input should be 1, 2 or 3
openedAt: Input should be a valid datetime or date, input is too short
car.manufacturer: String should have at least 2 characters
car.bodyType: Input should be 'sedan', 'coupe', 'suv' or 'truck'
car.manufacturedDate: Input should be greater than or equal to 1980-01-01
car.numberOfDoors: Input should be less than or equal to 4
car.registrationCountry.name: Value error, não pode conter dígitos
car.registrationCountry.code3: String should match pattern '^[A-Z]{3}$'
car.color: Extra inputs are not permitted


E o default inválido que o meta-modelo não pega:

In [9]:
broken_default = raw_spec | {"models": raw_spec["models"] | {"Extra": {"fields": {"doors": {"type": "int", "le": 4, "default": 9}}}}}
try:
    build_pydantic(SpecFile.model_validate(broken_default))
except ValueError as ex:
    print(f"ValueError: {ex}")

ValueError: Extra.doors: default 9 inválido: Input should be less than or equal to 4


## 4. Alvo 3 — dataclasses stdlib com `make_dataclass`

O mesmo descritor gera dataclasses comuns. Duas pegadinhas aparecem na hora:

In [10]:
try:
    make_dataclass("Bad", [("a", int, field(default=1)), ("b", int)])
except TypeError as ex:
    print(f"ordem dos campos: {ex}")
try:
    make_dataclass("Bad", [("tags", list[str], field(default=[]))])
except ValueError as ex:
    print(f"default mutável: {ex}")

ordem dos campos: non-default argument 'b' follows default argument 'a'
default mutável: mutable default <class 'list'> for field tags is not allowed: use default_factory


- a ordem dos campos na especificação não garante "obrigatórios antes dos opcionais" → `kw_only=True` resolve;
- default `[]` é proibido em dataclass (seria compartilhado) → o construtor converte em `default_factory`.

A política vira `__pydantic_config__` (no namespace da classe), para quando a dataclass for validada pelo Pydantic:

In [11]:
def build_dataclasses(spec: SpecFile) -> dict[str, type]:
    namespace: dict[str, Any] = build_enums(spec)
    for model_name in spec.model_order():
        model = spec.models[model_name]
        policy = config_for(model.policy)
        strict = bool(policy.get("strict"))
        definitions = []
        for name, spec_field in model.fields.items():
            annotation = full_type(spec_field, namespace)
            if spec_field.required:
                definitions.append((name, Annotated[annotation, Strict()] if strict else annotation))
                continue
            default = checked_default(model_name, name, spec_field, annotation)
            if strict:                                     # strict por campo (ver nota abaixo)
                annotation = Annotated[annotation, Strict()]
            if isinstance(default, (list, dict, set)):
                definitions.append((name, annotation, field(default_factory=lambda d=default: type(d)(d))))
            else:
                definitions.append((name, annotation, field(default=default)))
        class_config = {k: v for k, v in policy.items() if k not in ("frozen", "strict")}
        namespace[model_name] = make_dataclass(
            model_name, definitions, kw_only=True, frozen=bool(policy.get("frozen")),
            namespace={"__doc__": model.description, "__pydantic_config__": class_config},
        )
    return namespace

Dois ajustes no construtor, cada um por um motivo verificado:
- `frozen` vai para o `make_dataclass` (quem manda é o decorador; repetir no config gera aviso — notebook 04);
- `strict` sai da configuração da **classe** e vira `Strict()` em **cada campo**. Com `strict=True` na classe, uma dataclass
  aninhada passa a exigir **instância exata** e recusa `dict` — diferente de `BaseModel`, que aceita `dict` mesmo estrito:

In [12]:
from dataclasses import dataclass


@dataclass
class StrictInner:
    __pydantic_config__ = {"strict": True}
    n: int


@dataclass
class Outer:
    inner: StrictInner


class StrictInnerModel(BaseModel):
    model_config = ConfigDict(strict=True)
    n: int


class OuterModel(BaseModel):
    inner: StrictInnerModel


try:
    TypeAdapter(Outer).validate_python({"inner": {"n": 1}})
except ValidationError as ex:
    print(f"dataclass estrita aninhada + dict: {ex.errors()[0]['type']}")
print(f"BaseModel estrito aninhado + dict: {OuterModel.model_validate({'inner': {'n': 1}})!r}")

dataclass estrita aninhada + dict: dataclass_exact_type
BaseModel estrito aninhado + dict: OuterModel(inner=StrictInnerModel(n=1))


In [13]:
dcs = build_dataclasses(spec)
dc_ticket = TypeAdapter(dcs["ServiceTicket"]).validate_python(ticket_data)
from dataclasses import is_dataclass
print(f"is_dataclass(dc_ticket.car): {is_dataclass(dc_ticket.car)}")
print(f"isinstance(dc_ticket.car, BaseModel): {isinstance(dc_ticket.car, BaseModel)}")
print(f"dc_ticket.car.manufacturer: {dc_ticket.car.manufacturer!r}")
print(f"mesmo resultado que o Pydantic: {asdict(dc_ticket) == ticket.model_dump()}")
print(f"construção direta NÃO valida: {dcs['RegistrationCountry'](name='1', code3='x')!r}")

is_dataclass(dc_ticket.car): True
isinstance(dc_ticket.car, BaseModel): False
dc_ticket.car.manufacturer: 'Bmw Motors'
mesmo resultado que o Pydantic: True
construção direta NÃO valida: RegistrationCountry(name='1', code3='x')


A dataclass gerada é uma dataclass stdlib de verdade (`asdict`, `fields`, `replace` funcionam) e o `TypeAdapter` aplica tudo: tipos,
restrições, validadores, aliases. Mas a construção direta não valida nada — a validação depende de quem constrói usar o adapter.

## 5. A conta do runtime: `pickle` e checagem estática

Classes criadas em runtime e guardadas num dicionário não têm um nome importável — e `pickle` (usado por `multiprocessing`,
caches, filas) precisa disso:

In [14]:
for label, obj in [("Pydantic (create_model)", ticket), ("dataclass (make_dataclass)", dc_ticket)]:
    try:
        pickle.dumps(obj)
        print(f"{label}: pickle ok")
    except (pickle.PicklingError, AttributeError) as ex:
        print(f"{label}: {type(ex).__name__}: {str(ex)[:90]}")

Pydantic (create_model): PicklingError: Can't pickle <class '__main__.ServiceTicket'>: attribute lookup ServiceTicket on __main__ 
dataclass (make_dataclass): PicklingError: Can't pickle <class '__main__.ServiceTicket'>: attribute lookup ServiceTicket on __main__ 


E para o mypy/pyright, `pyd["ServiceTicket"]` é só `type[Any]`/`type[BaseModel]`: nenhum campo, nenhum autocompletar (medido na
seção 7). As duas coisas se resolvem com o último alvo.

## 6. Alvo 4 — gerar código-fonte

O gerador emite um módulo Python comum: enums como classes, modelos Pydantic com `Annotated`, validadores importados do registro.
O mesmo `SpecFile` alimenta tudo, então a equivalência pode ser **testada**.

In [15]:
SCALAR_SOURCE = {"str": "str", "int": "int", "float": "float", "bool": "bool", "date": "date", "datetime": "datetime",
                 "decimal": "Decimal", "uuid4": "UUID4"}


def py_literal(value: Any) -> str:
    """repr que funciona com os imports do arquivo gerado (repr(date) seria 'datetime.date(...)', que quebra)."""
    if isinstance(value, (StrEnum, IntEnum)):
        return f"{type(value).__name__}.{value.name}"
    if isinstance(value, datetime):
        return f"datetime({value.year}, {value.month}, {value.day}, {value.hour}, {value.minute}, {value.second})"
    if isinstance(value, date):
        return f"date({value.year}, {value.month}, {value.day})"
    return repr(value)


def type_source(spec_field: FieldSpec) -> str:
    base = SCALAR_SOURCE.get(spec_field.type, spec_field.type)
    constraints = [f"{k}={py_literal(getattr(spec_field, k))}" for k in ("ge", "le", "min_length", "max_length", "pattern")
                   if getattr(spec_field, k) is not None]
    metadata = (
        ([f"Field({', '.join(constraints)})"] if constraints else [])
        + [f"BeforeValidator(NORMALIZERS[{n!r}])" for n in reversed(spec_field.normalize)]
        + [f"AfterValidator(VALIDATORS[{v!r}])" for v in spec_field.validators]
    )
    source = f"Annotated[{base}, {', '.join(metadata)}]" if metadata else base
    if spec_field.many:
        source = f"list[{source}]"
    if spec_field.nullable:
        source = f"{source} | None"
    return source


def config_source(policy: str) -> str:
    items = [f"{k}={v}" if k == "alias_generator" else f"{k}={v!r}"
             for k, v in ((k, ALIAS_GENERATORS[v].__name__ if k == "alias_generator" else v)
                          for k, v in POLICIES[policy].items())]
    return f"ConfigDict({', '.join(items)})"


def render(spec: SpecFile, source_name: str) -> str:
    lines = [
        f"# GERADO por 05_modelos_a_partir_de_especificacao.ipynb a partir de {source_name} — NÃO EDITAR À MÃO.",
        "# Para mudar um modelo, edite a especificação e regenere. Regra de negócio: subclasses em outro módulo.",
        "from datetime import date, datetime",
        "from decimal import Decimal",
        "from enum import IntEnum, StrEnum",
        "from typing import Annotated",
        "",
        "from pydantic import UUID4, AfterValidator, BaseModel, BeforeValidator, ConfigDict, Field",
        "from pydantic.alias_generators import to_camel",
        "",
        "from validadores import NORMALIZERS, VALIDATORS",
        "",
    ]
    for name, enum_spec in spec.enums.items():
        base = "IntEnum" if enum_spec.kind == "int" else "StrEnum"
        lines += ["", f"class {name}({base}):"] + [f"    {m} = {v!r}" for m, v in enum_spec.members().items()] + [""]
    for name in spec.model_order():
        model = spec.models[name]
        lines += ["", f"class {name}(BaseModel):"]
        if model.description:
            lines.append(f'    """{model.description}"""')
        lines += [f"    model_config = {config_source(model.policy)}", ""]
        for field_name, spec_field in model.fields.items():
            extras = [] if spec_field.required else [
                f"default={py_literal(checked_default(name, field_name, spec_field, full_type(spec_field, pyd)))}"]
            if spec_field.description:
                extras.append(f"description={spec_field.description!r}")
            rhs = f" = Field({', '.join(extras)})" if extras else ""
            lines.append(f"    {field_name}: {type_source(spec_field)}{rhs}")
        lines.append("")
    return "\n".join(lines).rstrip() + "\n"


generated_dir = Path("generated")
generated_dir.mkdir(exist_ok=True)
target = generated_dir / "modelos_gerados.py"
code = render(spec, "configs/models.yaml")
target.write_text(code)
print(f"{target}:\n{code}")

generated/modelos_gerados.py:
# GERADO por 05_modelos_a_partir_de_especificacao.ipynb a partir de configs/models.yaml — NÃO EDITAR À MÃO.
# Para mudar um modelo, edite a especificação e regenere. Regra de negócio: subclasses em outro módulo.
from datetime import date, datetime
from decimal import Decimal
from enum import IntEnum, StrEnum
from typing import Annotated

from pydantic import UUID4, AfterValidator, BaseModel, BeforeValidator, ConfigDict, Field
from pydantic.alias_generators import to_camel

from validadores import NORMALIZERS, VALIDATORS


class BodyType(StrEnum):
    SEDAN = 'sedan'
    COUPE = 'coupe'
    SUV = 'suv'
    TRUCK = 'truck'


class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3


class RegistrationCountry(BaseModel):
    model_config = ConfigDict(extra='forbid', frozen=True, strict=True)

    name: Annotated[str, Field(min_length=2), AfterValidator(VALIDATORS['no_digits'])]
    code3: Annotated[str, Field(pattern='^[A-Z]{3}$')]


class Automobile(

### Testes do código gerado

Um gerador sem testes é uma fonte de bugs que ninguém revisa. Três verificações:

In [16]:
import importlib
import sys

sys.path.insert(0, str(generated_dir))
generated = importlib.import_module("modelos_gerados")
importlib.reload(generated)

# 1) o arquivo no repo está atualizado (regenerar não muda nada) -> serve como teste de CI
print(f"arquivo em dia com a especificação: {target.read_text() == render(spec, 'configs/models.yaml')}")

# 2) mesmo contrato que o alvo de runtime: JSON Schema idêntico
same_schema = all(getattr(generated, n).model_json_schema() == pyd[n].model_json_schema() for n in spec.models)
print(f"JSON Schema igual ao create_model (3 modelos): {same_schema}")

# 3) mesmo comportamento: mesmo resultado no dado válido, mesmos erros no inválido
gen_ticket = generated.ServiceTicket.model_validate(ticket_data)
print(f"mesmo resultado no dado válido: {gen_ticket.model_dump(mode='json') == ticket.model_dump(mode='json')}")


def error_set(model, data):
    try:
        model.model_validate(data)
        return set()
    except ValidationError as ex:
        return {(e["loc"], e["type"]) for e in ex.errors()}


print(f"mesmos erros no dado inválido: {error_set(generated.ServiceTicket, bad_ticket) == error_set(pyd['ServiceTicket'], bad_ticket)}")
print(f"pickle do modelo gerado: {pickle.loads(pickle.dumps(gen_ticket)) == gen_ticket}")

arquivo em dia com a especificação: True
JSON Schema igual ao create_model (3 modelos): True
mesmo resultado no dado válido: True
mesmos erros no dado inválido: True
pickle do modelo gerado: True


## 7. Checagem estática: runtime × gerado (mypy + plugin do Pydantic)

In [17]:
import importlib.util
import os
import subprocess
import tempfile
import textwrap

USAGE_GENERATED = """
from datetime import date
from modelos_gerados import Automobile, BodyType

car = Automobile(manufacturer="BMW", body_type=BodyType.SEDAN, manufactured_date=date(2020, 1, 1))
reveal_type(car.number_of_doors)
car.number_of_doors.upper()
Automobile(manufacturer="BMW", body_type="van", manufactured_date=date(2020, 1, 1))
"""
USAGE_RUNTIME = """
from typing import Any
from pydantic import BaseModel

models: dict[str, type[BaseModel]] = {}
Automobile = models["Automobile"]
car = Automobile(manufacturer="BMW")
reveal_type(car)
car.number_of_doors.upper()
"""


def run_mypy(snippet: str) -> list[str]:
    if importlib.util.find_spec("mypy") is None:
        return ["(mypy não instalado: pip install mypy)"]
    with tempfile.TemporaryDirectory() as tmp:
        Path(tmp, "uso.py").write_text(textwrap.dedent(snippet))
        Path(tmp, "mypy.ini").write_text("[mypy]\nplugins = pydantic.mypy\nignore_missing_imports = True\n"
                                         "[pydantic-mypy]\ninit_typed = True\n")
        env = os.environ | {"MYPYPATH": os.pathsep.join([str(generated_dir.resolve()), str(Path.cwd())])}
        result = subprocess.run([sys.executable, "-m", "mypy", "--config-file", "mypy.ini", "--no-incremental", "--no-color-output", "uso.py"],
                                cwd=tmp, capture_output=True, text=True, env=env)
    return [line.replace("uso.py:", "linha ") for line in result.stdout.splitlines() if "uso.py" in line]


for label, snippet in [("gerado", USAGE_GENERATED), ("runtime", USAGE_RUNTIME)]:
    for line in run_mypy(snippet):
        print(f"{label}: {line}")

gerado: linha 6: note: Revealed type is "int"
gerado: linha 7: error: "int" has no attribute "upper"  [attr-defined]
gerado: linha 8: error: Argument "body_type" to "Automobile" has incompatible type "str"; expected "BodyType"  [arg-type]


runtime: linha 8: note: Revealed type is "pydantic.main.BaseModel"
runtime: linha 9: error: "BaseModel" has no attribute "number_of_doors"  [attr-defined]


- **Gerado:** o mypy conhece cada campo (`number_of_doors` é `int`, logo `.upper()` é erro) e a assinatura do construtor
  (`body_type="van"` não é `BodyType`). Atenção: a checagem do construtor só acontece com `init_typed = True` na seção
  `[pydantic-mypy]`; no padrão, o plugin tipa os argumentos do `__init__` como `Any`, porque o Pydantic aceita coerção (`"1"` para `int`).
- **Runtime:** o tipo é só `BaseModel`; o mypy não sabe que campos existem — erro de atributo **em qualquer** acesso, ou
  silêncio total se o tipo vier como `Any`.

## 8. Qual alvo usar?

| alvo | validação | tipagem estática | `pickle` | revisão no git | quando |
|---|---|---|---|---|---|
| `create_model` | ✅ | ❌ | ❌ (sem nome importável) | só o YAML | modelos que mudam sem deploy (ex.: formulários definidos pelo usuário) |
| `make_dataclass` + `TypeAdapter` | ✅ (se usar o adapter) | ❌ | ❌ | só o YAML | domínio sem Pydantic + validação na borda |
| Enum funcional | ✅ | ❌ | depende do nome | só o YAML | vocabulários vindos de dados |
| **código gerado** | ✅ | ✅ | ✅ | **YAML + código** | **padrão quando a especificação é a fonte da verdade** |

Para a maioria dos casos: gerar código, versionar o arquivo gerado, e ter no CI o teste "regenerar não muda nada" + equivalência.
E antes de construir o seu meta-modelo, considere o pronto: JSON Schema + `datamodel-codegen` (curso, seção 12).

In [18]:
assert spec.model_order().index("Automobile") < spec.model_order().index("ServiceTicket")
assert gen_ticket.priority == generated.Priority.MEDIUM
assert target.read_text() == render(spec, "configs/models.yaml")
print("verificações: ok")

verificações: ok
